# 05 - Business insights and recommendations

## Business insight summary

This notebook consolidates findings from notebooks 01-04 into a coherent operation playbook.

### Insight 1: Funnel biggest drop is "browse -> favorite" (event counts)
- 11.5M browse events vs 120K purchase events
- Browse -> favorite drops 97.90% (event counts)
- Only 1.04% of browse events lead to purchase

### Insight 2: 11.14% of users never purchase ("window shoppers")
- Compared to converting users, they have 1/4 the browse intensity, 1/3 the browse depth
- Their add-to-cart rate is 1.55x lower than converting users
- Attribution hypothesis: passive browsing without strong intent

### Insight 3: 8 RFM segments are highly heterogeneous
- Largest: Potential loyalists (23.8%)
- Champions (14.8%) are the core value base
- Hibernating high-value (11.4%) is the recall target

### Insight 4: D1 is the critical retention inflection point
- D0: 100%, D1: 51.58% (biggest single-day drop, -48.42%)
- After D3 retention stabilizes around 44-47%
- D1 is the key window for new-customer activation

### Insight 5: Late evening is the golden hour
- Browse and purchase both peak at 20-22
- 24-hour distribution suggests concentrated marketing is more efficient

## Project indicator system

Before diving into specific analyses, let me establish the metric system that anchors the entire project. **A good data analyst doesn't just calculate metrics — they design them**.

### North Star Metric

**Monthly Purchasing Active Users (MPAU)** — number of users who make at least one purchase in 30 days.

Why this metric?
- It captures **both engagement and conversion** in one number
- It is **leading** (predicts revenue, GMV) without requiring price data
- It is **actionable** (operations can directly influence it)

### Metric hierarchy (3 layers)

```
                    MPAU (North Star)
                   /       |        \
                  /        |         \
        Layer 1: Activity     Layer 2: Conversion     Layer 3: Quality
        (quantity)            (efficiency)              (depth)
        |                     |                        |
        - DAU browse           - Browse -> purchase     - RFM composite score
        - DAU purchase         - Add-to-cart rate       - Avg behaviors per user
        - New users            - D1 retention           - Category diversity
```

### Layer 1: Activity metrics (quantity)

| Metric | Value in this dataset | Business meaning |
|--------|----------------------|------------------|
| Total behaviors | 12.26M | Engagement volume |
| DAU browse | 10,000 | Daily active user base |
| DAU purchase | ~3,920 (120K/30) | Daily purchasers |
| Per-user behaviors (median) | 747 | Engagement intensity |

### Layer 2: Conversion metrics (efficiency)

| Metric | Value | Insight |
|--------|-------|---------|
| Browse -> purchase rate | 1.04% | Low overall conversion |
| Browse -> favorite drop | 97.90% | Biggest funnel bottleneck |
| D1 retention | 51.58% | Half of users churn after first day |
| D7 retention | 46.70% | Stabilizes after D3 |

### Layer 3: Quality metrics (depth)

| Metric | Value | Insight |
|--------|-------|---------|
| Median behaviors/user | 747 | Heavy users dominate |
| Median categories/user | 72 | Wide category engagement |
| Champion segment share | 14.8% | High-value users |
| Hibernating high-value | 11.4% | Recall opportunity |

### How metrics connect across notebooks

| Notebook | Layer | Metrics introduced |
|----------|-------|--------------------|
| 01 | Activity | DAU, behavior distribution, user activity |
| 02 | Conversion | Funnel rates, drop-off analysis |
| 03 | Quality | RFM scores, segment sizes |
| 04 | Conversion | D1/D3/D7 retention |
| 05 | (Summary) | Quantified impact, methodology |

This metric system is the **scaffolding** for every analysis in this project. Each notebook answers questions about specific layers.


## Anomaly attribution methodology

In real data analyst work, a common task is: **"an important metric suddenly changed (up or down). Find out why."** This section shows my systematic approach.

### The 4-step attribution framework

```
1. LOCATE   ->  Where? Which day? Which segment? Which dimension?
2. DECOMPOSE ->  Drill down by time/channel/segment/category
3. HYPOTHESIZE ->  List all plausible causes (external/internal)
4. VERIFY ->  Test each hypothesis with data
```

### Worked example: "Day 18 purchase rate dropped 30%"

Imagine it's Day 18 (Dec 5) and the daily purchase rate suddenly drops 30% vs the previous week. How would I approach it?

**Step 1 — LOCATE the anomaly**

Compute a 7-day rolling baseline, flag days where the metric deviates >2 standard deviations. Confirm it's not a single-day blip (could be system glitch).

**Step 2 — DECOMPOSE by dimensions**

Decompose the drop by:
- **Time**: hour-of-day? Specific user segment?
- **User segment**: which RFM group dropped most?
- **Category**: which categories dropped?
- **Funnel stage**: drop at browse? At cart? At checkout?

In our data, let's actually check Day 18 (the 30th day, the dataset end):

In [1]:
import sys
sys.path.append("../scripts")
import pandas as pd
import numpy as np
from utils import load_data

df = load_data()
df["date"] = df["time"].dt.date

# Daily purchase rate
daily = df.groupby("date").agg(
    browse=("behavior_type", lambda x: (x == 1).sum()),
    purchase=("behavior_type", lambda x: (x == 4).sum()),
).reset_index()
daily["purchase_rate"] = daily["purchase"] / daily["browse"] * 100

# 7-day rolling baseline
daily["rolling_mean"] = daily["purchase_rate"].rolling(7, min_periods=3).mean()
daily["rolling_std"] = daily["purchase_rate"].rolling(7, min_periods=3).std()
daily["z_score"] = (daily["purchase_rate"] - daily["rolling_mean"]) / daily["rolling_std"]

# Flag anomalies
daily["anomaly"] = daily["z_score"].abs() > 2
print(f"Days flagged as anomalies: {daily['anomaly'].sum()}")
print()
print("Top 5 most anomalous days (largest |z-score|):")
print(daily.nlargest(5, "z_score", keep="all")[["date", "purchase_rate", "rolling_mean", "z_score"]].to_string(index=False))
print()
print("Top 5 most anomalous days (smallest |z-score|):")
print(daily.nsmallest(5, "z_score", keep="all")[["date", "purchase_rate", "rolling_mean", "z_score"]].to_string(index=False))

Days flagged as anomalies: 1

Top 5 most anomalous days (largest |z-score|):
      date  purchase_rate  rolling_mean  z_score
2014-12-12       2.377371      1.071903 2.245972
2014-11-26       1.048966      0.996434 1.051843
2014-11-27       1.048452      0.997600 0.992869
2014-12-01       1.050538      1.003632 0.972154
2014-12-03       1.002588      0.990769 0.247096

Top 5 most anomalous days (smallest |z-score|):
      date  purchase_rate  rolling_mean   z_score
2014-12-11       0.700803      0.871333 -1.850365
2014-12-10       0.808729      0.911340 -1.669315
2014-12-06       0.891247      0.971338 -1.620821
2014-11-23       0.926580      1.024154 -1.476130
2014-12-07       0.864587      0.958710 -1.470158


**Real finding from this dataset**:

Running the same algorithm on our 30-day data flagged **exactly 1 day as anomaly**: **Dec 12 (12月12日)** — purchase rate spiked to 2.38% (z-score +2.25, vs 1.07% rolling mean).

This is **Double 12 (双12)** — Taobao's biggest annual shopping festival. The framework correctly identified the known business event without any external labels — proof that the methodology works.

This demonstrates: **a good anomaly attribution framework can surface known business events automatically**, freeing analyst time from manual monitoring.

**Step 3 — HYPOTHESIZE causes**

For any flagged day, generate hypotheses:

| Category | Hypothesis | Test method |
|----------|-----------|-------------|
| External | Holiday / special event | Check Chinese calendar |
| External | Marketing campaign end | Check campaign calendar |
| Internal | Product bug / outage | Check tech log |
| Internal | Operation action | Check operation log |
| Data | Data pipeline issue | Check ETL log |
| Natural | Weekend vs weekday | Day-of-week comparison |

**Step 4 — VERIFY each hypothesis**

Run a quick test for each:
- **Holiday**: compare Dec 24 (Christmas eve) vs Dec 17 — if Dec 24 is anomalous, holiday hypothesis survives
- **Data issue**: compare to "ground truth" (e.g., GMV report from finance) — if mismatch, data hypothesis survives
- **Operation action**: ask operation team — if they ran a campaign that day, confirmed

### Why this framework matters

Without a framework, analysts often:
- Jump to conclusions ("users must be churning!")
- Look at the wrong dimension ("let me check traffic sources")
- Spend hours on the wrong hypothesis

The 4-step framework forces **discipline**: locate first, decompose second, hypothesize third, verify fourth.

In this project's Notebook 02, we already used a simpler version: we decomposed the funnel drop-off by **drop-off vs converting users** (step 2+3) and proposed "choice overload" as hypothesis (step 3), then verified by checking browse intensity and depth (step 4).


## Quantified business impact (illustrative)

> ⚠️ Numbers below are **illustrative projections** based on the funnel structure in this dataset. They assume the project is scaled to a typical e-commerce scenario (10K MAU, 500K daily browse events). They are NOT precise predictions.

**Scenario assumptions**:
- MAU: 10,000
- Daily browse events: 500,000
- Funnel rates extrapolated from this dataset

### Projection 1: If "browse -> favorite" lifts 2 percentage points

Current event-count funnel: browse 100% -> favorite 2.10% -> cart 2.97% -> purchase 1.04%
If browse -> favorite lifts to 4.10% (still conservative):
- Additional favorite events: 500K x 2% = 10,000 / day
- Following cart -> purchase rate (~35%): 10,000 x 35% = 3,500 / day in additional purchases
- **Monthly additional orders estimate: 105,000**

### Projection 2: If D1 recall lifts D1 retention by 5 percentage points

If D1 retention lifts from 51.58% to 56.58%:
- Additional active users on D1: 10,000 x 5% = 500 / day
- Assume these users contribute 1 purchase over the next 30 days: 500 / day x 30 = 15,000 / month
- **Monthly additional orders estimate: 15,000**

### Projection 3: If we reactivate 10% of hibernating high-value

Hibernating high-value segment has 1,138 users (11.4% of base).
If 10% are reactivated: 114 users
Assume each contributes 5 events (1 purchase) over the next month: 114 purchases
- **Monthly additional orders estimate: ~114 (smaller scale)**

> Note: these projections are based on multiple assumptions and are for **demonstrating that analysis can quantify business impact**, not for precise forecasting.

## Data limitations

Honestly stating the boundaries of this project:

1. **30-day time window**: retention analysis can only see 30 days of decay; cannot judge long-term retention (D60, D90, D180).
2. **Anonymized user/item/category IDs**: cannot interpret business meaning of categories (e.g., "apparel vs electronics"); analysis can only rank by ID.
3. **No price information**: cannot calculate GMV, AOV, ARPPU — all conclusions center on **behavior**, not revenue.
4. **No user attributes**: gender/age/location/device all missing — no user profile segmentation.
5. **No marketing/operation records**: cannot attribute retention/conversion changes to specific campaigns.
6. **10K user sample**: this is 1/100 of the original 1M-user dataset; some cohort-level statistics have small-sample noise.

## Future extensions

With richer data, this could become:
1. **Price-dimension analysis**: AOV, GMV contribution, high-value categories
2. **User-attribute segmentation**: gender/age/geography conversion differences
3. **A/B test attribution**: link retention/conversion changes to specific operations
4. **Predictive modeling**: build a purchase-propensity model (data-science direction)
5. **Real-time analytics**: streaming pipeline for real-time user behavior
6. **Recommendation system**: from "choice overload" finding, build a personalized recommendation prototype

## Chart index

All charts are in `images/` directory, organized by notebook:

| Notebook | Charts |
|---------|--------|
| 01 - Data exploration | behavior distribution, 24-hour distribution, user activity distribution |
| 02 - Funnel analysis | overall funnel, funnel by hour, funnel by category, attribution comparison |
| 03 - RFM segmentation | segment pie chart, segment radar chart |
| 04 - Retention analysis | retention heatmap, overall retention curve |

| File | Purpose |
|------|---------|
| `01_behavior_distribution.png` | 4 behavior types distribution |
| `01_hourly_distribution.png` | 24-hour behavior pattern |
| `01_user_activity_distribution.png` | User activity long-tail |
| `02_overall_funnel.png` | 4-step funnel |
| `02_funnel_by_hour.png` | Conversion rate by hour |
| `02_funnel_by_category.png` | Top 10 category conversion rates |
| `02_attribution_comparison.png` | Drop-off vs converting user behavior comparison |
| `03_rfm_segment_pie.png` | 8 RFM segment sizes |
| `03_rfm_segment_radar.png` | RFM profile comparison |
| `04_cohort_retention_heatmap.png` | 30-day cohort heatmap |
| `04_overall_retention_curve.png` | Overall retention curve |

In [2]:
import sys
sys.path.append('../scripts')
import pandas as pd

# Verify all derived data exists
from pathlib import Path
DERIVED = Path('../data/derived')

print("Derived datasets:")
for f in sorted(DERIVED.glob('*.csv')):
    df = pd.read_csv(f)
    print('  ' + f.name + ': ' + str(df.shape))


Derived datasets:
  cohort_retention.csv: (31, 35)
  rfm_segments.csv: (10000, 9)
